In [1]:
!apt-get update -qq
!apt-get install -y flex bison gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 78 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 

In [2]:
%%writefile backend.l
%{
#include "backend.tab.h"
#include <string.h>
#include <stdlib.h>
%}

%%

[a-zA-Z][a-zA-Z0-9]* {
    yylval.str = strdup(yytext);
    return ID;
}

"=" {
    return '=';
}

"+" {
    return '+';
}

"-" {
    return '-';
}

"*" {
    return '*';
}

"/" {
    return '/';
}

";" {
    return ';';
}

[ \t\n]+ {
    /* skip whitespace */
}

. {
    return yytext[0];
}

%%

int yywrap()
{
    return 1;
}

Writing backend.l


In [3]:
%%writefile backend.y
%{
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

int yylex(void);
int yyerror(char *s);
%}

%union {
    char *str;
}

%token <str> ID
%type <str> expr

%left '+' '-'
%left '*' '/'

%%

stmt_list:
      stmt_list stmt
    | stmt
    ;

stmt:
      ID '=' expr ';'
      {
          printf("MOV %s, AX\n\n", $1);
      }
    ;

expr:
      ID
      {
          printf("MOV AX, %s\n", $1);
          $$ = $1;
      }

    | expr '+' ID
      {
          printf("ADD AX, %s\n", $3);
          $$ = $3;
      }

    | expr '-' ID
      {
          printf("SUB AX, %s\n", $3);
          $$ = $3;
      }

    | expr '*' ID
      {
          printf("MUL %s\n", $3);
          $$ = $3;
      }

    | expr '/' ID
      {
          printf("MOV DX, 0\n");
          printf("MOV BX, %s\n", $3);
          printf("DIV BX\n");
          $$ = $3;
      }
    ;

%%

int main()
{
    printf("Enter TAC statements:\n");
    yyparse();
    return 0;
}

int yyerror(char *s)
{
    printf("Syntax Error: %s\n", s);
    return 0;
}

Writing backend.y


In [4]:
!rm -f backend.tab.c backend.tab.h lex.yy.c backend

In [5]:
!bison -d backend.y

In [6]:
!flex backend.l

In [7]:
!gcc lex.yy.c backend.tab.c -o backend -lfl

In [8]:
!ls -l backend

-rwxr-xr-x 1 root root 31952 Aug 25 14:08 backend


In [9]:
!echo "a=b;" | ./backend


Enter TAC statements:
MOV AX, b
MOV a, AX



In [10]:
!echo "a=b+c;" | ./backend

Enter TAC statements:
MOV AX, b
ADD AX, c
MOV a, AX



In [11]:
!echo "a=b-c;" | ./backend


Enter TAC statements:
MOV AX, b
SUB AX, c
MOV a, AX



In [12]:
!echo "a=b*c;" | ./backend

Enter TAC statements:
MOV AX, b
MUL c
MOV a, AX



In [13]:
!echo "a=b/c;" | ./backend

Enter TAC statements:
MOV AX, b
MOV DX, 0
MOV BX, c
DIV BX
MOV a, AX

